# B2-024-gpu-scientific-ml-capstone — Practice p20 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** mixture-parameter-regression

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

TRAIN_STEPS = 1500
REFINE_STEPS = 100
REFINE_LR = 0.02

## Solution

**Amortized regressor.** An MLP maps the 32 standardized samples (statistics from the training functions, stored as buffers) to the nine canonical parameters; training on canonical targets makes the target a genuine function of the input (practice p14).

**Refinement.** The regressor's guess is a *starting point* for per-function least squares: 100 Adam steps in float64 on $(w, c, \log s)$ (the log keeps widths positive), minimizing each function's mean squared residual over the 32 points. Because the loss is a sum over functions and Adam is elementwise, every function is refined independently. Starting near the right basin is what the fixed-initialization baseline lacks: from centers $0.25/0.5/0.75$ some fits get stuck with components in the wrong place.

**Pipeline.** `predict(values) = ls_refine(values, canonicalize(model(values)), 100)`.

In [ ]:
f, theta = capstone_data.train_rows("mixture")
fv, theta_v = capstone_data.val_rows("mixture")
X = capstone_data.MIXTURE_X


def mixture_function(x, weights, centers, widths):
    diff = x[None, :, None] - centers[:, None, :]
    return (weights[:, None, :] * torch.exp(-diff ** 2 / (2 * widths[:, None, :] ** 2))).sum(dim=-1)


def canonicalize(params):
    p = params.reshape(-1, 3, 3)
    order = torch.argsort(p[..., 1], dim=1, stable=True)
    return torch.gather(p, 1, order[..., None].expand(-1, -1, 3)).reshape(-1, 9)


class Regressor(nn.Module):
    def __init__(self, train_values, hidden=128):
        super().__init__()
        self.register_buffer("mu", train_values.mean(dim=0))
        self.register_buffer("sd", train_values.std(dim=0))
        self.net = nn.Sequential(nn.Linear(32, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 9))

    def forward(self, values):
        return self.net((values - self.mu) / self.sd)


SEAM = set()
seam_state = {"armed": True}


def seam_hook(module, args):
    if module.training and seam_state["armed"]:
        seam_state["armed"] = False
        SEAM.update(capstone_data.split_of("mixture", row) for row in args[0])


torch.manual_seed(SEED)
model = Regressor(f)
model.register_forward_pre_hook(seam_hook)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
model.train()
losses = []
for _ in range(TRAIN_STEPS):
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(f), theta)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
model.eval()


def ls_refine(values, init_params, steps, lr=REFINE_LR):
    p = init_params.detach().double().reshape(-1, 3, 3)
    w = p[..., 0].clone().requires_grad_(True)
    c = p[..., 1].clone().requires_grad_(True)
    log_s = torch.log(p[..., 2].clamp_min(1e-3)).requires_grad_(True)
    opt = torch.optim.Adam([w, c, log_s], lr=lr)
    x = X.double()
    target = values.detach().double()
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = ((mixture_function(x, w, c, torch.exp(log_s)) - target) ** 2).mean(dim=1).sum()
        loss.backward()
        opt.step()
    with torch.no_grad():
        out = torch.stack([w, c, torch.exp(log_s)], dim=-1).reshape(-1, 9)
    return canonicalize(out).float()


def raw_predict(values):
    model.eval()
    with torch.no_grad():
        return canonicalize(model(values))


def predict(values):
    return ls_refine(values, raw_predict(values), steps=REFINE_STEPS)


err = lambda pred: capstone_data.perm_invariant_error_per_example(pred, theta_v).mean().item()
val_raw = raw_predict(fv)
val_pipe = predict(fv)
val_base = capstone_data.mixture_ls_fit(fv, capstone_data.mixture_baseline_init(200), steps=500)
VAL = {"raw regressor": err(val_raw), "pipeline (regressor + 100 LS steps)": err(val_pipe), "baseline recipe (500 LS steps)": err(val_base)}
print(f"training loss {losses[0]:.4f} -> {losses[-1]:.5f} (ratio {losses[-1] / losses[0]:.4f}) | seam {SEAM}")
for name, e in VAL.items():
    print(f"validation perm-invariant error, {name}: {e:.4f}")

**The raw regressor loses to the baseline on validation** (about $0.024$ against $0.009$).
An 800-function MLP gives a smooth, averaged estimate: its parameter errors are dominated by the limited training set and by small-scale details of each bump (exact width and center) that a per-function least-squares fit recovers directly from that function's 32 samples.
Per-function least squares, when it starts in the right basin, fits each function to the noise level; the regressor is valuable not as the final answer but as that starting point, which is why the pipeline beats both.

**Locked test score of the pipeline (one call, after all fitting).**

In [ ]:
baseline = capstone_data.baseline_score("mixture")
assert capstone_data.test_call_count("mixture") == 0
test = capstone_data.final_test_score(predict, n_boot=1000, alpha=0.05, task="mixture")
assert capstone_data.test_call_count("mixture") == 1
print(f"pipeline: test error {test.score:.4f}, 95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}]")
print(f"committed baseline: {baseline.score:.4f} [{baseline.ci_low:.4f}, {baseline.ci_high:.4f}]")

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

On a Colab L4, scale the same regressor and refinement code with a replacement config: simulate many more training functions (`capstone_data.simulate("mixture", 500000, seed=4)`), widen the network, train with mini-batches, and keep the 100-step refinement (in float32 on the GPU, which is fast enough there).

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "simulated_functions": 500000, "hidden": 512,
             "steps": 30000, "batch_size": 4096, "lr": 1e-3, "refine_steps": 100, "refine_dtype": torch.float32}
```

**What to observe.** Whether the raw regressor overtakes the per-function baseline once it has hundreds of thousands of training functions; how much refinement still adds; and the effect of `bfloat16` autocast on the regressor's parameter precision (refinement should stay in `float32`).

*Documentation only: this section is not executed by CI; the CPU configuration above is the certified path.*

### Answer check

In [ ]:
assert len(losses) == 1500
assert losses[-1] <= 0.05 * losses[0]
assert SEAM == {"train"}
assert val_pipe.shape == (200, 9) and bool((val_pipe.reshape(-1, 3, 3)[..., 2] > 0).all())
assert VAL["pipeline (regressor + 100 LS steps)"] < VAL["baseline recipe (500 LS steps)"]
assert VAL["raw regressor"] > VAL["baseline recipe (500 LS steps)"]      # stated honestly above
assert capstone_data.test_call_count("mixture") == 1
assert test.score < baseline.score